# 1. Install dependencies
Install all necessary libraries for training, LoRA, and ONNX export.


In [52]:
!pip install -q transformers datasets accelerate peft onnx onnxruntime onnxruntime-tools Pillow torch torchvision wandb "torchao>=0.16.0" onnxscript

# 2. Create local working directories
Setup local storage under `/content/custom_split_vlm/`.


In [53]:
import os

base_dir = '/content/custom_split_vlm/'
dirs = [
    'checkpoints/stage1_alignment',
    'checkpoints/stage2_instruction',
    'checkpoints/stage3_vqa',
    'exports',
    'data'
]
for d in dirs:
    os.makedirs(os.path.join(base_dir, d), exist_ok=True)
print(f"Created directories under {base_dir}")

Created directories under /content/custom_split_vlm/


# 3. Imports and global config


In [54]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoProcessor, AutoModel, AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
import numpy as np
import onnx
import onnxruntime as ort
from PIL import Image
import requests

# Global Config
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
VISION_MODEL_ID = 'google/siglip-base-patch16-224'
LLM_MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
IMAGE_SIZE = 224
NUM_PI_TOKENS = 64
COMPRESSED_DIM = 256
SIGLIP_HIDDEN_SIZE = 768
PROJECTOR_HIDDEN_SIZE = 2048

print(f"Using device: {DEVICE}, dtype: {DTYPE}")

Using device: cuda, dtype: torch.bfloat16


# 4. Load SigLIP processor and vision model


In [55]:
print("Loading SigLIP processor and model...")
processor = AutoProcessor.from_pretrained(VISION_MODEL_ID)
base_vision_model = AutoModel.from_pretrained(VISION_MODEL_ID, attn_implementation="eager").vision_model
base_vision_model.eval()
print("SigLIP loaded.")

Loading SigLIP processor and model...


Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

SigLIP loaded.


# 5. Define SplitSigLIPVision class
Splits SigLIP after block 2 (blocks 0 and 1 on Pi). Pools to 64 tokens and compresses.


In [56]:
class SplitSigLIPVision(nn.Module):
    def __init__(self, base_vision_model):
        super().__init__()
        self.embeddings = base_vision_model.embeddings
        self.pi_layers = base_vision_model.encoder.layers[:2]
        self.backend_layers = base_vision_model.encoder.layers[2:]
        self.post_layernorm = base_vision_model.post_layernorm

        # Compression/Decompression
        self.compressor = nn.Linear(SIGLIP_HIDDEN_SIZE, COMPRESSED_DIM)
        self.decompressor = nn.Linear(COMPRESSED_DIM, SIGLIP_HIDDEN_SIZE)

        # Fixed SigLIP 224/16 token pool, ONNX-friendly equivalent of adaptive average pooling
        pool_weights = torch.zeros(196, NUM_PI_TOKENS)
        for i in range(NUM_PI_TOKENS):
            start, end = i * 196 // NUM_PI_TOKENS, ((i + 1) * 196 + NUM_PI_TOKENS - 1) // NUM_PI_TOKENS
            pool_weights[start:end, i] = 1 / (end - start)
        self.register_buffer('pool_weights', pool_weights)

    def encode_pi_side(self, pixel_values):
        # Embeddings (Patch + Position)
        hidden_states = self.embeddings(pixel_values)
        # Blocks 0 and 1
        for layer_module in self.pi_layers:
            # attention_mask is a required positional argument, pass None explicitly
            hidden_states = layer_module(hidden_states, None)

        # Pool to 64 tokens (B, Seq, Dim) -> (B, Dim, Seq) -> Pool -> (B, Seq, Dim)
        hidden_states = hidden_states.transpose(1, 2).matmul(self.pool_weights).transpose(1, 2)

        # Compress
        compressed_tokens = self.compressor(hidden_states)
        return compressed_tokens

    def decode_backend_side(self, compressed_tokens):
        # Decompress
        hidden_states = self.decompressor(compressed_tokens)
        # Remaining blocks
        for layer_module in self.backend_layers:
            # attention_mask is a required positional argument, pass None explicitly
            hidden_states = layer_module(hidden_states, None)

        hidden_states = self.post_layernorm(hidden_states)
        return hidden_states

    def forward_full(self, pixel_values):
        comp = self.encode_pi_side(pixel_values)
        return self.decode_backend_side(comp)

    def export_pi_side_onnx(self, path='/content/custom_split_vlm/exports/pi_side_siglip_split.onnx'):
        class PiSide(nn.Module):
            def __init__(self, parent):
                super().__init__()
                self.parent = parent
            def forward(self, pixel_values):
                return self.parent.encode_pi_side(pixel_values)

        import copy
        pi_model = PiSide(copy.deepcopy(self).cpu().float()).eval()
        dummy_input = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE, dtype=torch.float32)

        torch.onnx.export(
            pi_model, dummy_input, path,
            export_params=True,
            opset_version=14,
            do_constant_folding=True,
            input_names=['pixel_values'],
            output_names=['compressed_tokens'],
            dynamic_axes={'pixel_values': {0: 'batch_size'}, 'compressed_tokens': {0: 'batch_size'}},
            dynamo=False # Fallback to TorchScript exporter
        )
        print(f"ONNX model exported to {path}")

split_vision_model = SplitSigLIPVision(base_vision_model).to(DEVICE, DTYPE)

# 6. Load Qwen/Qwen2.5-1.5B-Instruct


In [57]:
print("Loading Qwen LLM...")
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
qwen_model = AutoModelForCausalLM.from_pretrained(LLM_MODEL_ID, torch_dtype=DTYPE).to(DEVICE)
QWEN_HIDDEN_SIZE = qwen_model.config.hidden_size
print(f"Qwen loaded. Hidden size: {QWEN_HIDDEN_SIZE}")

Loading Qwen LLM...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen loaded. Hidden size: 1536


# 7. Define projector
2-layer MLP to map from SigLIP hidden (768) to Qwen hidden size.


In [58]:
class Projector(nn.Module):
    def __init__(self):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(SIGLIP_HIDDEN_SIZE, PROJECTOR_HIDDEN_SIZE),
            nn.GELU(),
            nn.Linear(PROJECTOR_HIDDEN_SIZE, QWEN_HIDDEN_SIZE)
        )
    def forward(self, x):
        return self.proj(x)

projector = Projector().to(DEVICE, DTYPE)

# 8. Define CustomSplitVLM class


In [59]:
class CustomSplitVLM(nn.Module):
    def __init__(self, vision_model, projector, llm, tokenizer):
        super().__init__()
        self.vision_model = vision_model
        self.projector = projector
        self.llm = llm
        self.tokenizer = tokenizer
        self.ignore_index = -100

    def encode_image_pi_side(self, pixel_values):
        return self.vision_model.encode_pi_side(pixel_values)

    def run_backend_vision(self, compressed_tokens):
        return self.vision_model.decode_backend_side(compressed_tokens)

    def project_visual_tokens(self, visual_features):
        return self.projector(visual_features)

    def _prepare_inputs(self, input_ids, attention_mask, pixel_values=None, compressed_tokens=None, labels=None):
        text_embeds = self.llm.get_input_embeddings()(input_ids)

        if pixel_values is not None or compressed_tokens is not None:
            if pixel_values is not None:
                comp = self.encode_image_pi_side(pixel_values)
            else:
                comp = compressed_tokens

            vis_features = self.run_backend_vision(comp)
            vis_embeds = self.project_visual_tokens(vis_features)

            # Prep final embeds by concat
            inputs_embeds = torch.cat([vis_embeds, text_embeds], dim=1)

            # Extend attention mask
            b = inputs_embeds.shape[0]
            vis_mask = torch.ones((b, NUM_PI_TOKENS), dtype=attention_mask.dtype, device=attention_mask.device)
            attention_mask = torch.cat([vis_mask, attention_mask], dim=1)

            if labels is not None:
                vis_labels = torch.full((b, NUM_PI_TOKENS), self.ignore_index, dtype=labels.dtype, device=labels.device)
                labels = torch.cat([vis_labels, labels], dim=1)
        else:
            inputs_embeds = text_embeds

        return inputs_embeds, attention_mask, labels

    def forward(self, input_ids, attention_mask, pixel_values=None, compressed_tokens=None, labels=None):
        inputs_embeds, attention_mask, labels = self._prepare_inputs(
            input_ids, attention_mask, pixel_values, compressed_tokens, labels
        )
        return self.llm(inputs_embeds=inputs_embeds, attention_mask=attention_mask, labels=labels)

    def generate(self, input_ids, attention_mask, pixel_values=None, compressed_tokens=None, max_new_tokens=128):
        inputs_embeds, attention_mask, _ = self._prepare_inputs(
            input_ids, attention_mask, pixel_values, compressed_tokens
        )
        return self.llm.generate(inputs_embeds=inputs_embeds, attention_mask=attention_mask, max_new_tokens=max_new_tokens)

vlm = CustomSplitVLM(split_vision_model, projector, qwen_model, tokenizer)

# 9. Load training datasets
Image-caption data is reused for all three stages with separate train/eval splits so the notebook can run a complete Colab training pass.


In [ ]:
STAGE_DATASET_ID = 'astro21/coco-caption-train-split-10k'
train_dataset = load_dataset(STAGE_DATASET_ID, split='train[:90%]').shuffle(seed=42)
eval_dataset = load_dataset(STAGE_DATASET_ID, split='train[90%:]').shuffle(seed=42)
stage1_dataset, stage1_eval_dataset = train_dataset, eval_dataset
stage2_dataset, stage2_eval_dataset = train_dataset, eval_dataset
stage3_dataset, stage3_eval_dataset = train_dataset, eval_dataset
print(f'Training rows: {len(train_dataset)}, eval rows: {len(eval_dataset)}')


# 10. Define dataset preprocessing
Each collator builds a small chat transcript, preprocesses images, masks padding labels, and returns tensors for the custom VLM forward pass.


In [ ]:
def build_text(prompt, answer):
    return tokenizer.apply_chat_template([{'role': 'user', 'content': prompt}, {'role': 'assistant', 'content': answer}], tokenize=False)

def collate_prompt(batch, prompt):
    images = [item['image'].convert('RGB') for item in batch]
    texts = [build_text(prompt, item['caption']) for item in batch]
    inputs = processor(images=images, return_tensors='pt')
    text_inputs = tokenizer(texts, padding=True, truncation=True, max_length=256, return_tensors='pt')
    labels = text_inputs['input_ids'].clone()
    labels[labels == tokenizer.pad_token_id] = -100
    return {'pixel_values': inputs['pixel_values'].to(DTYPE), 'input_ids': text_inputs['input_ids'], 'attention_mask': text_inputs['attention_mask'], 'labels': labels}

def collate_stage1(batch): return collate_prompt(batch, 'Describe this image.')
def collate_stage2(batch): return collate_prompt(batch, 'Describe this image in detail.')
def collate_stage3(batch): return collate_prompt(batch, 'Question: What is shown in this image?')


# 11. Stage 1 training: alignment
Freeze SigLIP and Qwen, then train only the compressor, decompressor, and visual projector.


In [ ]:
print('Training Stage 1 alignment...')
for p in vlm.parameters(): p.requires_grad = False
for module in [vlm.vision_model.compressor, vlm.vision_model.decompressor, vlm.projector]:
    for p in module.parameters(): p.requires_grad = True

training_args_stage1 = TrainingArguments(
    output_dir='/content/custom_split_vlm/checkpoints/stage1_alignment',
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=5e-4,
    warmup_ratio=0.03,
    weight_decay=0.01,
    lr_scheduler_type='cosine',
    max_grad_norm=1.0,
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
    logging_steps=25,
    eval_strategy='steps',
    eval_steps=250,
    save_strategy='steps',
    save_steps=250,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    report_to='none',
    remove_unused_columns=False,
)
trainer_stage1 = Trainer(model=vlm, args=training_args_stage1, train_dataset=stage1_dataset, eval_dataset=stage1_eval_dataset, data_collator=collate_stage1)
trainer_stage1.train()
print('Stage 1 complete.')


# 12. Stage 2 training: instruction tuning
Attach LoRA adapters to Qwen and train the adapters plus the visual projector on image-description prompts.


In [ ]:
print('Training Stage 2 instruction tuning...')
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM'
)
vlm.llm = get_peft_model(vlm.llm, lora_config)
for p in vlm.vision_model.parameters(): p.requires_grad = False
for p in vlm.projector.parameters(): p.requires_grad = True
for n, p in vlm.llm.named_parameters(): p.requires_grad = 'lora_' in n

training_args_stage2 = TrainingArguments(
    output_dir='/content/custom_split_vlm/checkpoints/stage2_instruction',
    num_train_epochs=2,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-5,
    warmup_ratio=0.03,
    weight_decay=0.01,
    lr_scheduler_type='cosine',
    max_grad_norm=1.0,
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
    logging_steps=25,
    eval_strategy='steps',
    eval_steps=250,
    save_strategy='steps',
    save_steps=250,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    report_to='none',
    remove_unused_columns=False,
)
trainer_stage2 = Trainer(model=vlm, args=training_args_stage2, train_dataset=stage2_dataset, eval_dataset=stage2_eval_dataset, data_collator=collate_stage2)
trainer_stage2.train()
print('Stage 2 complete.')


# 13. Stage 3 training: VQA tuning
Continue training the LoRA adapters and projector with question-style image prompts.


In [ ]:
print('Training Stage 3 VQA tuning...')
for p in vlm.vision_model.parameters(): p.requires_grad = False
for p in vlm.projector.parameters(): p.requires_grad = True
for n, p in vlm.llm.named_parameters(): p.requires_grad = 'lora_' in n

training_args_stage3 = TrainingArguments(
    output_dir='/content/custom_split_vlm/checkpoints/stage3_vqa',
    num_train_epochs=2,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=1e-5,
    warmup_ratio=0.03,
    weight_decay=0.01,
    lr_scheduler_type='cosine',
    max_grad_norm=1.0,
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
    logging_steps=25,
    eval_strategy='steps',
    eval_steps=250,
    save_strategy='steps',
    save_steps=250,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    report_to='none',
    remove_unused_columns=False,
)
trainer_stage3 = Trainer(model=vlm, args=training_args_stage3, train_dataset=stage3_dataset, eval_dataset=stage3_eval_dataset, data_collator=collate_stage3)
trainer_stage3.train()
print('Stage 3 complete.')


# 14. Export Pi-side model only


In [65]:
split_vision_model.export_pi_side_onnx()

/tmp/ipykernel_1000/1635644554.py:63: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


ValueError: not enough values to unpack (expected 3, got 2)

# 15. Raspberry Pi ONNX inference example


In [ ]:
import onnxruntime as ort
import numpy as np

session = ort.InferenceSession('/content/custom_split_vlm/exports/pi_side_siglip_split.onnx')

def run_pi_inference(image_path):
    image = Image.open(image_path).convert('RGB')
    inputs = processor(images=image, return_tensors="np")
    pixel_values = inputs["pixel_values"].astype(np.float32)

    ort_inputs = {session.get_inputs()[0].name: pixel_values}
    ort_outs = session.run(None, ort_inputs)

    compressed_tokens = ort_outs[0]
    np.save('/content/custom_split_vlm/data/visual_tokens.npy', compressed_tokens)
    return compressed_tokens

print("Pi inference code defined.")

# 16. Backend inference from image


In [ ]:
def backend_inference_image(image_path, prompt):
    image = Image.open(image_path).convert('RGB')
    inputs = processor(images=image, return_tensors="pt")
    pixel_values = inputs["pixel_values"].to(DEVICE, dtype=DTYPE)

    text_inputs = tokenizer(prompt, return_tensors='pt').to(DEVICE)

    with torch.no_grad():
        outputs = vlm.generate(
            input_ids=text_inputs.input_ids,
            attention_mask=text_inputs.attention_mask,
            pixel_values=pixel_values,
            max_new_tokens=50
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Backend inference (image) defined.")

# 17. Backend inference from saved compressed tokens


In [ ]:
def backend_inference_tokens(tokens_path, prompt):
    compressed_tokens_np = np.load(tokens_path)
    compressed_tokens = torch.from_numpy(compressed_tokens_np).to(DEVICE, dtype=DTYPE)

    text_inputs = tokenizer(prompt, return_tensors='pt').to(DEVICE)

    with torch.no_grad():
        outputs = vlm.generate(
            input_ids=text_inputs.input_ids,
            attention_mask=text_inputs.attention_mask,
            compressed_tokens=compressed_tokens,
            max_new_tokens=50
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Backend inference (tokens) defined.")